Taking the plots out of the file ev_experiments to safe the data 

In [263]:
from __future__ import annotations

import os
from typing import Callable, Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib
import glob
import pickle
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

# from ev_core import (
#     EVStagHuntModel,
#     set_initial_adopters,
#     final_mean_adoption_vs_ratio,
#     phase_sweep_X0_vs_ratio,
# )
# from ev_plotting import (
#     plot_fanchart,
#     plot_spaghetti,
#     plot_density,
#     plot_ratio_sweep,
#     plot_phase_plot,
    
# )

from ev_experiments import (
     plot_intervention_fanchart,
     
     
     
#     phase_sweep_df,
#     ratio_sweep_df
    

)

Pickle 

In [264]:
def load_all_saved_objects(base_dir="."):
    """
    Walk through base_dir, find all *_data.pkl inside data_{label} folders,
    and load the full dict from each pickle (arrays, DataFrames, whatever).

    Returns:
        data[label][dataset_name] = dict_from_pickle
    """
    data = {}

    for root, _, files in os.walk(base_dir):
        for fname in files:
            if not fname.endswith("_data.pkl"):
                continue

            path = os.path.join(root, fname)

            folder = os.path.basename(root)
            if not folder.startswith("data_"):
                continue  # adjust if you're still using plots_{label}

            label = folder.replace("data_", "")
            dataset_name = fname.replace("_data.pkl", "")

            with open(path, "rb") as f:
                obj = pickle.load(f)

            if label not in data:
                data[label] = {}
            data[label][dataset_name] = obj

    return data

In [265]:
if __name__ == "__main__":
    all_dfs = load_all_saved_objects()

    # Example: access some specific DataFrames
    # all_dfs["InitialAdoption0.3"]["intervention"]["baseline_df"]
    # all_dfs["InitialAdoption0.3"]["phase"]["phase_df"]
    # all_dfs["InitialAdoption0.3"]["ratio_sweep"]["sweep_df"]

    # Quick sanity print
    for label, datasets in all_dfs.items():
        print(f"\n=== {label} ===")
        for name, df_dict in datasets.items():
            for key, df in df_dict.items():
                print(f"{name}.{key}")


=== BA ===
intervention.baseline_X
intervention.baseline_I
intervention.subsidy_X
intervention.subsidy_I
intervention.baseline_df
intervention.subsidy_df
phase.phase_df
ratio_sweep.sweep_df
spaghetti_density.traces_df

=== ER ===
intervention.baseline_X
intervention.baseline_I
intervention.subsidy_X
intervention.subsidy_I
intervention.baseline_df
intervention.subsidy_df
phase.phase_df
ratio_sweep.sweep_df
spaghetti_density.traces_df

=== Grids ===
intervention.baseline_X
intervention.baseline_I
intervention.subsidy_X
intervention.subsidy_I
intervention.baseline_df
intervention.subsidy_df
phase.phase_df
ratio_sweep.sweep_df
spaghetti_density.traces_df

=== HighBetaI3.0 ===
intervention.baseline_X
intervention.baseline_I
intervention.subsidy_X
intervention.subsidy_I
intervention.baseline_df
intervention.subsidy_df
phase.phase_df
ratio_sweep.sweep_df
spaghetti_density.traces_df

=== InitialAdoption0.3 ===
intervention.baseline_X
intervention.baseline_I
intervention.subsidy_X
intervention

In [ ]:
label = "InitialAdoption0.3"
scenario_data = all_dfs[label]

# Unpack each dataset into variables named after their keys
for dataset_name, contents in scenario_data.items():
    for key, value in contents.items():
        globals()[key] = value


In [267]:
# label = "InitialAdoption0.5"
# scenario_data = all_dfs[label]

# # Unpack each dataset into variables named after their keys
# for dataset_name, contents in scenario_data.items():
#     for key, value in contents.items():
#         globals()[key] = value

In [268]:
# label = "InitialInfrastructur0.15"
# scenario_data = all_dfs[label]

# # Unpack each dataset into variables named after their keys
# for dataset_name, contents in scenario_data.items():
#     for key, value in contents.items():
#         globals()[key] = value

In [269]:
# label = "InitialInfrastructur0.25"
# scenario_data = all_dfs[label]

# # Unpack each dataset into variables named after their keys
# for dataset_name, contents in scenario_data.items():
#     for key, value in contents.items():
#         globals()[key] = value

In [270]:
# label = "HighBetaI3.0"
# scenario_data = all_dfs[label]

# # Unpack each dataset into variables named after their keys
# for dataset_name, contents in scenario_data.items():
#     for key, value in contents.items():
#         globals()[key] = value

In [ ]:
# label = "LowBetaI1.0"
# scenario_data = all_dfs[label]

# # Unpack each dataset into variables named after their keys
# for dataset_name, contents in scenario_data.items():
#     for key, value in contents.items():
#         globals()[key] = value

In [272]:
print(baseline_df.shape)
print(len(baseline_X))
print(traces_df.head())
print(phase_df.shape)
print(sweep_df.head())

(300, 2)
200
      group  trial  time    X
0  baseline      0     0  0.0
1  baseline      0     1  0.0
2  baseline      0     2  0.0
3  baseline      0     3  0.0
4  baseline      0     4  0.0
(651, 3)
   ratio  X_mean
0   0.80     0.0
1   0.89     0.0
2   0.98     0.0
3   1.07     0.0
4   1.16     0.0


Create directories

In [273]:
folder = f"plots_{label}"
os.makedirs(folder, exist_ok=True)

Fanchart

In [274]:
def plot_fanchart(traces_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot fan charts (quantile bands) for baseline vs subsidy using traces DF.

    traces_df columns: ['group', 'trial', 'time', 'X'] where group in {'baseline','subsidy'}.
    """
    if traces_df.empty:
        raise ValueError("traces_df is empty")

    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]

        # Compute quantiles by time across trials
        q = gdf.groupby("time")["X"].quantile([0.10, 0.25, 0.75, 0.90]).unstack(level=1)
        mean = gdf.groupby("time")["X"].mean()
        t = mean.index.to_numpy()

        ax = axes[0, j]
        ax.fill_between(t, q[0.10], q[0.90], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.15, label="10–90%")
        ax.fill_between(t, q[0.25], q[0.75], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.30, label="25–75%")

        # Overlay some traces for context (sample up to 100 trials)
        trial_ids = gdf["trial"].unique()
        rng = np.random.default_rng(123)
        sample = rng.choice(trial_ids, size=min(100, len(trial_ids)), replace=False)
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.1, linewidth=0.8)

        ax.plot(t, mean, color=("steelblue" if group == "baseline" else "darkorange"), linewidth=2, label="mean")
        ax.set_title(f"{group.capitalize()} adoption")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)
        ax.legend(loc="lower right")

        # Final X(T) histogram
        t_max = int(gdf["time"].max())
        final_vals = gdf[gdf["time"] == t_max].groupby("trial")["X"].mean().to_numpy()
        axes[1, j].hist(final_vals, bins=20, color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.8)
        axes[1, j].set_title(f"{group.capitalize()} final X(T)")
        axes[1, j].set_xlabel("X(T)")
        axes[1, j].set_ylabel("Count")

    if out_path is None:
        out_path = _default_plot_path("ev_intervention_fanchart.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

In [275]:
fanchart_path = plot_intervention_fanchart(
    baseline_X,
    subsidy_X,
    out_path=f"plots_{label}/fanchart_{label}.png", 
)
print("Saved fanchart image:", fanchart_path)
        

Saved fanchart image: plots_LowBetaI1.0/fanchart_LowBetaI1.0.png


Phase 

In [276]:
def plot_phase_plot(phase_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot heatmap from tidy DataFrame with columns ['X0','ratio','X_final']."""
    # Pivot to matrix for imshow
    pivot = phase_df.pivot(index="ratio", columns="X0", values="X_final").sort_index().sort_index(axis=1)
    ratios = pivot.index.to_numpy()
    X0s = pivot.columns.to_numpy()

    plt.figure(figsize=(7, 4))
    im = plt.imshow(
        pivot.to_numpy(),
        origin="lower",
        extent=[X0s[0], X0s[-1], ratios[0], ratios[-1]],
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
        cmap="plasma",
    )
    plt.colorbar(im, label="Final adopters X*")
    plt.xlabel("X0 (initial adoption)")
    plt.ylabel("a_I / b (initial payoff ratio)")
    plt.title("Network phase plot: X* over X0 and a_I/b")

 # Overlay threshold X = 1/ratio
    X_thresh = 1.0 / ratios
    X_thresh_clipped = np.clip(X_thresh, 0.0, 1.0)
    plt.plot(X_thresh_clipped, ratios, color="white", linestyle="--", linewidth=1.5, label="X = b / a_I (initial)")
    plt.legend(loc="upper right")

    if out_path is None:
        out_path = _default_plot_path("ev_phase_plot.png")
    plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close()
    return out_path

In [277]:
# Also run the phase plot of X* over (X0, a_I/b) and save it
phase_path = plot_phase_plot(phase_df, out_path=f"plots_{label}/phase_{label}.png")
print("Saved phase plot:", phase_path)

Saved phase plot: plots_LowBetaI1.0/phase_LowBetaI1.0.png


Spaghetti 

In [278]:
def plot_spaghetti(traces_df: pd.DataFrame, *, max_traces: int = 100, alpha: float = 0.15, out_path: Optional[str] = None) -> str:
    """Spaghetti plot from traces DF for baseline vs subsidy."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    rng = np.random.default_rng(123)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        trial_ids = gdf["trial"].unique()
        sample = rng.choice(trial_ids, size=min(max_traces, len(trial_ids)), replace=False)
        ax = axes[j]
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=alpha, linewidth=0.8)
        ax.set_title(f"{group.capitalize()} traces")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)

    if out_path is None:
        out_path = _default_plot_path("ev_spaghetti.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path



In [279]:
def plot_density(traces_df: pd.DataFrame, *, x_bins: int = 50, time_bins: Optional[int] = None, out_path: Optional[str] = None) -> str:
    """Time-evolving density plot (2D histogram) from traces DF."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        T = int(gdf["time"].max()) + 1
        if time_bins is None:
            bins_time = T
        else:
            bins_time = time_bins
        hb = axes[j].hist2d(gdf["time"].to_numpy(), gdf["X"].to_numpy(), bins=[bins_time, x_bins], range=[[0, T - 1], [0.0, 1.0]], cmap="magma")
        axes[j].set_title(f"{group.capitalize()} density: time vs X(t)")
        axes[j].set_xlabel("Time")
        axes[j].set_ylabel("X(t)")
        fig.colorbar(hb[3], ax=axes[j], label="count")

    if out_path is None:
        out_path = _default_plot_path("ev_density.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

In [280]:
# Spaghetti and time-evolving density plots
spaghetti_path = plot_spaghetti(traces_df, max_traces=100, alpha=0.15, out_path=f"plots_{label}/spaghetti_{label}.png")
        
print("Saved spaghetti plot:", spaghetti_path)

density_path = plot_density(traces_df, x_bins=50, time_bins=200, out_path=f"plots_{label}/density_{label}.png")
print("Saved time-evolving density plot:", density_path)


Saved spaghetti plot: plots_LowBetaI1.0/spaghetti_LowBetaI1.0.png
Saved time-evolving density plot: plots_LowBetaI1.0/density_LowBetaI1.0.png


Ratio Sweep 

In [281]:
def plot_ratio_sweep(sweep_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot X* vs ratio from a DataFrame with columns ['ratio','X_mean']."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sweep_df["ratio"], sweep_df["X_mean"], color="C0", lw=2)
    ax.set_xlabel("a_I / b (ratio)")
    ax.set_ylabel("Final adoption X*")
    ax.set_title("X* vs ratio")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.25)
    if out_path is None:
        out_path = _default_plot_path("ev_ratio_sweep.png")
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return out_path

In [282]:
ratio_sweep_path=plot_ratio_sweep(sweep_df, out_path=f"plots_{label}/ratio_sweep_{label}.png")
print("Saved ratio sweep plot:", ratio_sweep_path)

Saved ratio sweep plot: plots_LowBetaI1.0/ratio_sweep_LowBetaI1.0.png
